# PillCheck — encoder training (Kaggle GPU)

One-time setup before running this:
1. Zip `ml/scripts`, `ml/train`, `ml/eval`, `ml/configs` (skip `ml/data`, `ml/outputs`, `.venv`) and upload as a Kaggle Dataset, e.g. named `pillcheck-ml`.
2. Attach that dataset to this notebook (Add Input).
3. Notebook settings: Accelerator = GPU, Internet = On.

`/kaggle/input` is read-only, so the first cell copies the uploaded code to a writable working copy — everything after that runs exactly like it does locally, no path changes needed.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!pip install -q timm pytorch-metric-learning

In [ ]:
# Change 'pillcheck-ml' to whatever you named the uploaded dataset.
!rm -rf /kaggle/working/ml
!cp -r /kaggle/input/pillcheck-ml /kaggle/working/ml
%cd /kaggle/working/ml

In [ ]:
!cd scripts && python download_epillid.py

In [ ]:
# Full run. Increase --epochs from configs/encoder.yaml's default if the first
# run finishes well within a session and the eval number still has headroom.
# If a session times out mid-run, re-run this cell with --resume-from outputs/encoder/latest.pt
# (re-attach this notebook's own prior output as an input dataset first, or re-copy from /kaggle/working
# if the session persisted).
!cd train && python encoder.py

## After training

`outputs/encoder/latest.pt` is the checkpoint. "Save Version" to persist `/kaggle/working` as this notebook's output, then download `ml/outputs/encoder/latest.pt` from the notebook's Output tab and place it at `ml/outputs/encoder/latest.pt` in the local repo.

Then, locally:
```
uv run python scripts/build_reference_index.py --checkpoint outputs/encoder/latest.pt
uv run python eval/harness.py --checkpoint outputs/encoder/latest.pt
```